# ANKA — Early Corpus, Tokenizer, and Binary-Shard Pipeline

> **Status:** Historical run_01 data pipeline

## Purpose

Preserve the early 2.4B-token corpus workflow, tokenizer training, corpus archiving, and binary-shard conversion used before the final English 5B run.

## Execution guidance

This notebook contains two historical corpus branches that write to the same run_01 paths. Do not run both branches sequentially unless replacement is intentional. The released English RMW3 checkpoint was produced by the later run_02 pipeline, not this notebook.

## Primary outputs

Corpus files, tokenizer.json, validation artifacts, and fixed uint16 shards.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Prepare the run_01 workspace

Initializes the runtime, deterministic settings, dependencies, and persistent paths required by later sections.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/ANKA-50M
!mkdir -p /content/ANKA-50M

!unzip -q \
  /content/drive/MyDrive/ANKA/ANKA-50M-full-pipeline.zip \
  -d /content/ANKA-50M

%cd /content/ANKA-50M

!pip install -q -r requirements.txt

## 2. Build the original mixed-language corpus

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
!python -m anka.corpus \
  --output /content/drive/MyDrive/ANKA/run_01/corpus \
  --target-tokens 2400000000 \
  --english-fraction 0.90

## 3. Train the original 15,360-token tokenizer

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
!python -m anka.tokenizer_pipeline \
  --corpus /content/drive/MyDrive/ANKA/run_01/corpus \
  --output /content/drive/MyDrive/ANKA/run_01/tokenizer \
  --vocab-size 15360

## 4. Archive the mixed-language corpus

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
from pathlib import Path
import shutil

CORPUS_DIR = Path(
    "/content/drive/MyDrive/ANKA/run_01/corpus"
)

ARCHIVE_BASE = Path(
    "/content/drive/MyDrive/ANKA/run_01/ANKA_corpus_2.4B"
)

assert CORPUS_DIR.exists(), f"Corpus not found: {CORPUS_DIR}"

archive_path = shutil.make_archive(
    base_name=str(ARCHIVE_BASE),
    format="zip",
    root_dir=str(CORPUS_DIR.parent),
    base_dir=CORPUS_DIR.name,
)

archive_size_gb = Path(archive_path).stat().st_size / 2**30

print("Corpus archived.")
print("Location:", archive_path)
print(f"ZIP size: {archive_size_gb:.2f} GB")

## 5. Reset the workspace for the Turkish-only branch

Initializes the runtime, deterministic settings, dependencies, and persistent paths required by later sections.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.


In [ ]:
!rm -rf /content/ANKA-50M
!mkdir -p /content/ANKA-50M

!unzip -q \
  /content/drive/MyDrive/ANKA/ANKA-50M-full-pipeline.zip \
  -d /content/ANKA-50M

%cd /content/ANKA-50M
!pip install -q -r requirements.txt

## 6. Build the Turkish-only corpus branch

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
!python -m anka.corpus \
  --output /content/drive/MyDrive/ANKA/run_01/corpus \
  --turkish-only \
  --turkish-target-chars 960000000

## 7. Retrain the tokenizer on the replacement corpus

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# Retrain after the replacement-corpus branch above.
!python -m anka.tokenizer_pipeline \
  --corpus /content/drive/MyDrive/ANKA/run_01/corpus \
  --output /content/drive/MyDrive/ANKA/run_01/tokenizer \
  --vocab-size 15360

## 8. Replace the run_01 corpus archive

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.


In [ ]:
from pathlib import Path
import shutil

old_zip = Path(
    "/content/drive/MyDrive/ANKA/run_01/ANKA_corpus_2.4B.zip"
)

if old_zip.exists():
    old_zip.unlink()

archive = shutil.make_archive(
    "/content/drive/MyDrive/ANKA/run_01/ANKA_corpus_2.4B",
    "zip",
    root_dir="/content/drive/MyDrive/ANKA/run_01",
    base_dir="corpus",
)

size_gb = Path(archive).stat().st_size / 2**30

print("Full corpus archived:", archive)
print(f"ZIP size: {size_gb:.2f} GB")

## 9. Tokenize the corpus into fixed binary shards

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
%cd /content/ANKA-50M

!python -m anka.data \
  --corpus /content/drive/MyDrive/ANKA/run_01/corpus \
  --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
  --output /content/drive/MyDrive/ANKA/run_01/shards \
  --shard-tokens 25000000 \
  --validation-fraction 0.005 \
  --encode-batch-size 512 \
  --seed 2026